# 05 - Avaliacao Clinica (conjunto de teste)\n\nCamada de decisao clinica. **Consome apenas artefactos committed** (`preds/*.npy`, `y_test.npy`, `results.csv`, `split_indices.npz`) -- sem dataset, sem modelos, sem GPU, sem Drive. Corre num portatil a partir do repositorio git.


In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# Camada de decisao clinica: le APENAS artefactos committed (preds/*.npy,
# y_test.npy, results.csv, split_indices.npz). Sem dataset, sem modelos, sem GPU.
# Corre localmente a partir do repositorio git; bootstrap.setup() nao monta o
# Drive fora do Colab.
try:
    from src import bootstrap, data, viz
except ModuleNotFoundError:
    import os, sys
    sys.path.insert(0, os.getcwd())
    from src import bootstrap, data, viz

IN_COLAB = bootstrap.setup()
REPO_DIR = bootstrap.get_repo_dir(IN_COLAB)
ARTIFACT_DIRS = bootstrap.get_artifact_dirs(REPO_DIR)
ARTIFACTS_DIR = ARTIFACT_DIRS["artifacts"]
PREDS_DIR = ARTIFACT_DIRS["preds"]


In [ ]:
# Registo de previsoes reconstruido a partir de artefactos committed: a tabela de
# resultados da a lista de modelos; para cada um carrega-se a matriz de
# probabilidades de teste (preds/{slug}_proba.npy). Nenhuma imagem, nenhum modelo.
df_results = pd.read_csv(ARTIFACTS_DIR / "results.csv")
y_test = np.load(PREDS_DIR / "y_test.npy")
class_names = data.CLASS_NAMES

probas = {}
for nome in df_results["Modelo"]:
    probas[nome] = np.load(PREDS_DIR / f"{data.slug(nome)}_proba.npy")

print("Modelos:", list(probas))
print("y_test:", y_test.shape)


## 4.3 Matriz de confusão e classification report
A matriz de confusão permite analisar em detalhe os erros do modelo por classe,
distinguindo entre **Falsos Positivos** (FP) e **Falsos Negativos** (FN).

No contexto do **Use Case 3 — Sistema de Referenciação Hospitalar**, os erros não
têm o mesmo custo clínico:

- **Falso Negativo** (doente com patologia classificado como Normal) — o caso mais
  grave: o doente não é encaminhado e a patologia fica por diagnosticar.
- **Falso Positivo** (exame normal enviado para especialista) — indesejável mas
  menos grave: gera carga desnecessária no sistema, mas sem consequências para o doente.

A análise da matriz de confusão no conjunto de teste — nunca visto durante o treino —
é a avaliação mais honesta da capacidade de generalização do modelo.

In [ ]:
# Matriz de confusao + classification report no TESTE (variante test-only: so ha
# previsoes de teste committed; a comparacao treino-vs-teste vive nos produtores).
for nome, proba in probas.items():
    print("\n" + "="*55)
    viz.plot_test_confusion(proba, y_test, class_names, nome_modelo=nome)


## 4.4 ROC
### Curva ROC (Receiver Operating Characteristic)
A curva ROC representa a relação entre a **Sensibilidade** (Taxa de Verdadeiros Positivos)
e **1 - Especificidade** (Taxa de Falsos Positivos) para diferentes thresholds de decisão.
O valor de **AUC-ROC** (Área Sob a Curva) sintetiza o desempenho global do modelo —
um valor de 1.0 representa classificação perfeita e 0.5 equivale a uma decisão aleatória.

### Curva Precision-Recall
A curva Precision-Recall é particularmente relevante em datasets desbalanceados como
o nosso, onde a classe *cataract* representa a minoria. Ao contrário da curva ROC,
não é influenciada pelo elevado número de verdadeiros negativos, sendo uma avaliação
mais honesta do desempenho nas classes minoritárias.

O valor de **Average Precision (AP)** resume a curva — quanto mais próximo de 1.0,
melhor o modelo deteta a classe em questão independentemente do threshold escolhido.

No contexto do **Use Case 3**, um AUC-ROC elevado na classe *cataract* é crítico:
garante que o modelo consegue separar doentes com catarata de casos normais,
reduzindo o risco de falsos negativos no encaminhamento hospitalar.

In [ ]:
for nome, proba in probas.items():
    print("\n" + "="*55)
    print(f"ROC e Precision-Recall -- {nome}")
    viz.plot_roc_pr_curves(proba, y_test, class_names, nome)


## 4.5 Sensibilidade vs Especificidade por Threshold

A curva de Sensibilidade vs Especificidade por threshold permite analisar o impacto
da escolha do threshold de decisão no desempenho do modelo para cada classe.

- **Sensibilidade (Recall)** — proporção de casos positivos corretamente identificados.
  No UC3, uma sensibilidade elevada nas classes patológicas garante que doentes com
  catarata ou retinopatia diabética não são incorretamente classificados como normais.
- **Especificidade** — proporção de casos negativos corretamente identificados.
  Uma especificidade elevada reduz o número de exames normais enviados desnecessariamente
  para avaliação especializada.

O threshold padrão de 0.5 nem sempre é o mais adequado num contexto clínico. A análise
desta curva permite identificar o ponto de equilíbrio entre sensibilidade e especificidade
para cada classe, informando uma possível calibração do sistema para o contexto hospitalar.

In [ ]:
for nome, proba in probas.items():
    print("\n" + "="*55)
    print(f"Sensibilidade vs Especificidade -- {nome}")
    viz.plot_sensitivity_specificity_threshold(proba, y_test, class_names, nome)


## 4.6 Tabela Comparativa de Modelos

In [ ]:
# Tabela comparativa ja persistida por 02/03/04 (artifacts/results.csv).
df_results.set_index("Modelo")


---\n## Trabalho posterior (placeholders)


## Justificacao do threshold de decisao

*Placeholder de um batch posterior -- sem conteudo nesta fase.*


## Reagrupamento binario: patologico vs normal

*Placeholder de um batch posterior -- sem conteudo nesta fase.* (Colapsar
Cataract + Diabetic Retinopathy numa classe "patologico" e reavaliar no teste.)


## Quantificacao de impacto clinico

*Placeholder de um batch posterior -- sem conteudo nesta fase.* (Exames
referenciados, normais evitados, Falsos Positivos, Falsos Negativos.)


## Discussao de leakage

*Placeholder de um batch posterior -- sem conteudo nesta fase.*
